In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [2]:
df=pd.read_csv("index_name_titanic.csv")

In [4]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [6]:
titanic_df=df[["Survived","Pclass","Sex","Age","SibSp","Parch","Fare","Embarked"]]

In [8]:
titanic_df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S
887,1,1,female,19.0,0,0,30.0000,S
888,0,3,female,NaN,1,2,23.4500,S
889,1,1,male,26.0,0,0,30.0000,C


In [10]:
x=titanic_df.drop("Survived",axis=1)
y=titanic_df[["Survived"]]
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [12]:
x_train.head(1)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S


In [13]:
trf=ColumnTransformer([("age_imp",SimpleImputer(),["Age"]),
                       ("embarked_imp",SimpleImputer(strategy="most_frequent"),["Embarked"]),
                       ("ohe_sex_embarked",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),["Sex","Embarked"])],remainder="passthrough")

In [15]:
pipe=Pipeline([("trf",trf)])

In [17]:
pipe.fit_transform(x_train)

array([[45.5, 'S', 0.0, ..., 0.0, 0.0, 28.5],
       [23.0, 'S', 0.0, ..., 0.0, 0.0, 13.0],
       [32.0, 'S', 0.0, ..., 0.0, 0.0, 7.925],
       ...,
       [41.0, 'S', 0.0, ..., 2.0, 0.0, 14.1083],
       [14.0, 'S', 1.0, ..., 1.0, 2.0, 120.0],
       [21.0, 'S', 0.0, ..., 0.0, 1.0, 77.2875]],
      shape=(712, 12), dtype=object)

In [19]:
pipe.get_feature_names_out()  #same as index number

array(['age_imp__Age', 'embarked_imp__Embarked',
       'ohe_sex_embarked__Sex_female', 'ohe_sex_embarked__Sex_male',
       'ohe_sex_embarked__Embarked_C', 'ohe_sex_embarked__Embarked_Q',
       'ohe_sex_embarked__Embarked_S', 'ohe_sex_embarked__Embarked_nan',
       'remainder__Pclass', 'remainder__SibSp', 'remainder__Parch',
       'remainder__Fare'], dtype=object)

In [20]:
embarked_pipe=Pipeline([("embarked_imp",SimpleImputer(strategy="most_frequent")),
                        ("embarked_ohe",OneHotEncoder(sparse_output=False,handle_unknown="ignore"))])

In [22]:
trf=ColumnTransformer([("age_imp",SimpleImputer(),["Age"]),
                       ("ohe_sex",OneHotEncoder(sparse_output=False,handle_unknown="ignore"),["Sex"]),
                       ("embarked_pipe",embarked_pipe,["Embarked"])],remainder="passthrough")

In [24]:
pipe=Pipeline([("trf",trf)])

In [26]:
pipe.fit_transform(x_train)

array([[ 45.5   ,   0.    ,   1.    , ...,   0.    ,   0.    ,  28.5   ],
       [ 23.    ,   0.    ,   1.    , ...,   0.    ,   0.    ,  13.    ],
       [ 32.    ,   0.    ,   1.    , ...,   0.    ,   0.    ,   7.925 ],
       ...,
       [ 41.    ,   0.    ,   1.    , ...,   2.    ,   0.    ,  14.1083],
       [ 14.    ,   1.    ,   0.    , ...,   1.    ,   2.    , 120.    ],
       [ 21.    ,   0.    ,   1.    , ...,   0.    ,   1.    ,  77.2875]],
      shape=(712, 10))

In [28]:
pipe.get_feature_names_out()

array(['age_imp__Age', 'ohe_sex__Sex_female', 'ohe_sex__Sex_male',
       'embarked_pipe__Embarked_C', 'embarked_pipe__Embarked_Q',
       'embarked_pipe__Embarked_S', 'remainder__Pclass',
       'remainder__SibSp', 'remainder__Parch', 'remainder__Fare'],
      dtype=object)

In [29]:
pd.DataFrame(pipe.fit_transform(x_train),columns=pipe.get_feature_names_out(),index=x_train.index)

,age_imp__Age,ohe_sex__Sex_female,ohe_sex__Sex_male,embarked_pipe__Embarked_C,embarked_pipe__Embarked_Q,embarked_pipe__Embarked_S,remainder__Pclass,remainder__SibSp,remainder__Parch,remainder__Fare
331,45.500000,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,28.5000
733,23.000000,0.0,1.0,0.0,0.0,1.0,2.0,0.0,0.0,13.0000
382,32.000000,0.0,1.0,0.0,0.0,1.0,3.0,0.0,0.0,7.9250
704,26.000000,0.0,1.0,0.0,0.0,1.0,3.0,1.0,0.0,7.8542
813,6.000000,1.0,0.0,0.0,0.0,1.0,3.0,4.0,2.0,31.2750
...,...,...,...,...,...,...,...,...,...,...
106,21.000000,1.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,7.6500
270,29.498846,0.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,31.0000
860,41.000000,0.0,1.0,0.0,0.0,1.0,3.0,2.0,0.0,14.1083
435,14.000000,1.0,0.0,0.0,0.0,1.0,1.0,1.0,2.0,120.0000
